In [1]:
import sys
import pandas as pd
sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr
import metatlas2.load_tools as ldt
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa
import metatlas2.rt_align_tools as rat
import metatlas2.targeted_analysis as tga
import metatlas2.targeted_gui as tgi
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
REPORT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}_comprehensive_report.xlsx'

In [ ]:
new_main_database = True
new_atlases = True
new_rt_alignment = True
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

Overwriting existing main database (overwrite_existing=True)
Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/20 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/65 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/85 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/373 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/418 [00:00<?, ?it/s]

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-948261b5838e415f9feb3ef62e19e0f6"
    TARGET_ATLAS_UID = "atl-raw-4239aecf4f51488f9b16b9329d5f8f31"


In [6]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-58ded23f08ba48b2abce38a078b8a9b4"

Categorizing files:   0%|          | 0/8 [00:00<?, ?it/s]

Extracting MS1 data:   0%|          | 0/2 [00:00<?, ?it/s]

Matching compounds:   0%|          | 0/20 [00:00<?, ?it/s]

,compound_name,inchi_key,ref_rt_peak,exp_rt_median,rt_diff_median,observation_count,exp_rt_std
0,guanine (U - 15N),UYTPUPDQBNUYGX-CIKZIQIKSA-N,6.2654,6.210600,-0.0548,2,0.0090
1,"glutamic acid (U - 13C, 15N)",WHUUTDBJXJRKMK-UFLWJPECSA-N,15.9354,15.966100,0.0307,2,0.0090
2,"aspartic acid (U - 13C, 15N)",CKLJMWTZIZZHCS-NYTNKSBQSA-N,16.1304,16.169500,0.0391,2,0.0116
3,"methionine (U - 13C, 15N)",FFEARJCKVFRZRR-XAFSXMPTSA-N,10.4410,10.409400,-0.0315,2,0.0075
4,"alanine (U - 13C, 15N)",QNAYBMKLOCPYGJ-UVYXLFMMSA-N,13.4051,13.358000,-0.0471,2,0.0095
5,adenine (U - 15N),GFFGJBXGBJISGV-CIKZIQIKSA-N,2.6776,2.816400,0.1388,2,0.0213
6,"cystine (U - 13C, 15N)",LEVWYRKDKASIDU-OGYFDXEDSA-N,16.9043,17.003099,0.0988,2,0.0066
7,"thymine (U - 13C, 15N)",RWQNBRDOKXIBIV-BNUYUSEDSA-N,1.2552,1.353900,0.0986,2,0.0003
8,hypoxanthine (U - 15N),FDGQSTZJBFJUBT-NNZQUYKOSA-N,3.1030,3.149900,0.0469,2,0.0215
9,ABMBA (unlabeled),LCMZECCEEOQWLQ-UHFFFAOYSA-N,1.0938,1.253400,0.1596,2,0.0001


In [7]:
# for inchi, inchi_data in project_data.compounds.items():
#     if inchi_data.ms2_data_files:
#         print(f'MS2 data for {inchi_data.inchi_key}:')
#         for inchi_data_ms2_file, inchi_data_ms2_data in inchi_data.ms2_data_files.items():
#             print(inchi_data_ms2_data['ms2_entries'][0].keys())
#             if inchi_data_ms2_data['best_hit']:
#                 print(inchi_data_ms2_file)
#                 print(inchi_data_ms2_data['best_hit'])
#             else:
#                 print(f'No MS2 hits found for {inchi_data_ms2_file}')
#     else:
#         print(f'No MS2 data for {inchi_data.inchi_key}')

In [8]:
if new_analysis is True:
    atlas_df_ft, project_data = tga.run_targeted_analysis_workflow(PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG)
    gui = tgi.create_gui(project_data, CONFIG)
    display(gui)

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]


=== DEBUG: create_plot_only for ABMBA (unlabeled) ===
RT bounds: 0.927 - 1.527, peak: 1.227
MS2 result: type=None, file=None
Query spec: False
Ref spec: False


In [9]:
if new_analysis is True:
    plot_data = gui.get_plot_data()
    post_analysis_atlas_uid = tga.create_post_analysis_atlas(PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, gui, CONFIG)
    targeted_analysis_uid = dbi.deposit_targeted_analysis_from_plot_data(atlas_df_ft, PROJECT_DB_PATH, plot_data, post_analysis_atlas_uid, PROJECT_NAME)
    comprehensive_report = dbi.generate_comprehensive_targeted_analysis_report(PROJECT_DB_PATH, CONFIG, targeted_analysis_uid, atlas_df_ft, REPORT_PATH)

AttributeError: module 'metatlas2.targeted_analysis' has no attribute 'create_post_analysis_atlas'